# 02: Full-Universe News Fetch (One-Shot, No Resume)

This notebook fetches news for the full stock universe (S&P 500+ symbols from price data) in one run.

Key rules:
- No resume logic
- No state-based continuation
- No old cache fallback
- Overwrite canonical output CSV each run

Canonical output:
- `project_folder/news_data/data/news_headlines_raw.csv`

In [1]:
%pip install -q pandas numpy requests yfinance kagglehub

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import time
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests
import yfinance as yf
import kagglehub

pd.set_option('display.max_colwidth', 120)
np.random.seed(42)

def resolve_news_root():
    cwd = Path.cwd()
    if cwd.name == 'implementation' and cwd.parent.name == 'news_data':
        return cwd.parent
    if cwd.name == 'news_data':
        return cwd
    if (cwd / 'project_folder' / 'news_data').exists():
        return cwd / 'project_folder' / 'news_data'
    if (cwd.parent / 'project_folder' / 'news_data').exists():
        return cwd.parent / 'project_folder' / 'news_data'
    if (cwd / 'news_data').exists():
        return cwd / 'news_data'
    if (cwd.parent / 'news_data').exists():
        return cwd.parent / 'news_data'
    return cwd / 'news_data'

news_root = resolve_news_root()
workspace_root = news_root.parent
data_dir = news_root / 'data'
data_dir.mkdir(parents=True, exist_ok=True)
raw_news_path = data_dir / 'news_headlines_raw.csv'
raw_full_path = data_dir / 'news_headlines_raw_full_unfiltered.csv'
log_path = data_dir / 'news_ingestion.log'
state_path = data_dir / 'news_ingestion_state.json'

# Hard reset behavior: remove old output/state before fetch.
for p in [raw_news_path, raw_full_path, state_path]:
    if p.exists():
        p.unlink()

cluster_data_dir = workspace_root / '03_stock_clustering_analysis' / 'data'
reg_data_dir = workspace_root / '02_stock_price_regression' / 'data'
price_candidates = [
    cluster_data_dir / 'sp500_raw.csv',
    reg_data_dir / 'sp500_raw.csv',
    reg_data_dir / '02C_sp500_raw.csv',
    reg_data_dir / '02B_sp500_raw.csv',
]

price_path = next((p for p in price_candidates if p.exists()), None)
if price_path is None:
    print('[INFO] Shared price cache not found locally; downloading reference S&P 500 dataset.')
    kaggle_path = kagglehub.dataset_download('camnugent/sandp500')
    csv_file = [f for f in os.listdir(kaggle_path) if f.endswith('.csv')][0]
    price_df = pd.read_csv(os.path.join(kaggle_path, csv_file))
else:
    price_df = pd.read_csv(price_path)

if 'Name' not in price_df.columns or 'date' not in price_df.columns:
    raise ValueError('Price data must contain Name and date columns.')

price_df = price_df.copy()
price_df['Name'] = price_df['Name'].astype(str).str.strip()
price_df['date'] = pd.to_datetime(price_df['date'], errors='coerce').dt.normalize()
price_df = price_df.dropna(subset=['Name', 'date']).sort_values(['Name', 'date']).reset_index(drop=True)

symbols = sorted(price_df['Name'].unique().tolist())
min_date = price_df['date'].min()
max_date = price_df['date'].max()

NEWS_BUFFER_DAYS = 30
window_start = (min_date - pd.Timedelta(days=NEWS_BUFFER_DAYS)).normalize()
window_end = max_date.normalize()

USE_NEWSAPI = True
USE_GOOGLE_RSS = True
USE_YFINANCE = True
NEWSAPI_KEY = os.getenv('NEWSAPI_KEY', '').strip()
use_newsapi_runtime = USE_NEWSAPI and bool(NEWSAPI_KEY)

MAX_NEWS_PER_STOCK = None
SLEEP_SEC = 0.05
LOG_EVERY = 20
GOOGLE_RSS_USE_DATE_FILTER = True

def log(msg):
    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    line = f'[{ts}] {msg}'
    print(line)
    with open(log_path, 'a', encoding='utf-8') as f:
        f.write(line + '\n')

def normalize_text(value):
    return ' '.join(str(value).split()).strip()

def parse_date(value):
    if value is None or value == '':
        return pd.NaT
    parsed = pd.to_datetime(value, utc=True, errors='coerce')
    if pd.isna(parsed):
        return pd.NaT
    return pd.to_datetime(parsed.date())

def safe_ts_to_date(ts):
    if ts is None or pd.isna(ts):
        return pd.NaT
    try:
        return pd.to_datetime(datetime.fromtimestamp(int(ts), tz=timezone.utc).date())
    except Exception:
        return parse_date(ts)

def fetch_yfinance_news(symbol, max_items=None):
    rows = []
    try:
        items = yf.Ticker(symbol.replace('.', '-')).news or []
    except Exception:
        items = []

    for item in items[:max_items]:
        title = normalize_text(item.get('title', ''))
        url = normalize_text(item.get('link', ''))
        seendate = safe_ts_to_date(item.get('providerPublishTime', None))
        if title and pd.notna(seendate):
            rows.append({
                'Name': symbol,
                'date': seendate,
                'headline': title,
                'source': 'yfinance',
                'url': url,
                'published_at': item.get('providerPublishTime', None),
                'query_symbol': symbol,
                'query_text': f'{symbol} stock',
            })
    return rows

def fetch_google_rss_news(symbol, max_items=None):
    rows = []
    try:
        if GOOGLE_RSS_USE_DATE_FILTER:
            q_txt = f'{symbol} stock after:{window_start.date()} before:{(window_end + pd.Timedelta(days=1)).date()}'
        else:
            q_txt = f'{symbol} stock'
        q = requests.utils.quote(q_txt)
        url = f'https://news.google.com/rss/search?q={q}&hl=en-US&gl=US&ceid=US:en'
        response = requests.get(url, timeout=15, headers={'User-Agent': 'Mozilla/5.0'})
        if not response.ok:
            return rows

        import xml.etree.ElementTree as ET
        root = ET.fromstring(response.content)
        items = root.findall('./channel/item')[:max_items]

        for item in items:
            title_node = item.find('title')
            link_node = item.find('link')
            pub_node = item.find('pubDate')
            title = normalize_text(title_node.text if title_node is not None and title_node.text else '')
            url = normalize_text(link_node.text if link_node is not None and link_node.text else '')
            pub_text = pub_node.text.strip() if pub_node is not None and pub_node.text else ''
            dts = pd.to_datetime(pub_text, utc=True, errors='coerce') if pub_text else pd.NaT
            seendate = pd.to_datetime(dts.date()) if pd.notna(dts) else pd.NaT
            if title and pd.notna(seendate):
                rows.append({
                    'Name': symbol,
                    'date': seendate,
                    'headline': title,
                    'source': 'google_rss',
                    'url': url,
                    'published_at': pub_text,
                    'query_symbol': symbol,
                    'query_text': q_txt,
                })
    except Exception:
        return rows
    return rows

def fetch_newsapi_news(symbol, api_key, page_size=30):
    if not api_key:
        return []

    rows = []
    try:
        params = {
            'q': f'{symbol} stock',
            'language': 'en',
            'sortBy': 'publishedAt',
            'pageSize': page_size,
            'from': str(window_start.date()),
            'to': str(window_end.date()),
            'apiKey': api_key
        }
        response = requests.get('https://newsapi.org/v2/everything', params=params, timeout=15)
        payload = response.json() if response.ok else {}
        articles = payload.get('articles', [])
    except Exception:
        articles = []

    for article in articles:
        title = normalize_text(article.get('title', ''))
        url = normalize_text(article.get('url', ''))
        published_at = article.get('publishedAt', None)
        seendate = parse_date(published_at)
        if title and pd.notna(seendate):
            rows.append({
                'Name': symbol,
                'date': seendate,
                'headline': title,
                'source': 'newsapi',
                'url': url,
                'published_at': published_at,
                'query_symbol': symbol,
                'query_text': f'{symbol} stock',
            })
    return rows

def fetch_symbol_news(symbol):
    rows = []
    counts = {'yfinance': 0, 'google_rss': 0, 'newsapi': 0}

    if USE_YFINANCE:
        y = fetch_yfinance_news(symbol, max_items=MAX_NEWS_PER_STOCK)
        rows.extend(y)
        counts['yfinance'] = len(y)

    if USE_GOOGLE_RSS:
        r = fetch_google_rss_news(symbol, max_items=MAX_NEWS_PER_STOCK)
        rows.extend(r)
        counts['google_rss'] = len(r)

    if use_newsapi_runtime:
        n = fetch_newsapi_news(symbol, NEWSAPI_KEY, page_size=30)
        rows.extend(n)
        counts['newsapi'] = len(n)

    return rows, counts

log('=== Full-universe one-shot news ingestion started ===')
log(f'Universe size: {len(symbols)} symbols')
log(f'Price coverage: {min_date.date()} -> {max_date.date()}')
log(f'News coverage target: {window_start.date()} -> {window_end.date()}')
log(f'NEWS_BUFFER_DAYS={NEWS_BUFFER_DAYS}')
log(f'USE_NEWSAPI={USE_NEWSAPI}, API key provided={bool(NEWSAPI_KEY)}, effective NewsAPI fetch={use_newsapi_runtime}')
log(f'USE_GOOGLE_RSS={USE_GOOGLE_RSS}, USE_YFINANCE={USE_YFINANCE}')
log(f'Output CSV: {raw_news_path}')

all_rows = []
loop_start = time.time()

for idx, symbol in enumerate(symbols, start=1):
    symbol_rows, source_counts = fetch_symbol_news(symbol)
    all_rows.extend(symbol_rows)

    if (idx % LOG_EVERY == 0) or (idx == len(symbols)):
        elapsed = time.time() - loop_start
        avg_sec = elapsed / max(1, idx)
        remaining = len(symbols) - idx
        eta_sec = int(avg_sec * max(0, remaining))
        eta_time = datetime.now() + pd.Timedelta(seconds=eta_sec)
        eta_str = eta_time.strftime('%Y-%m-%d %H:%M:%S')
        log(
            f'Progress {idx}/{len(symbols)} | current={symbol} | got={len(symbol_rows)} '
            f"(yf={source_counts['yfinance']}, rss={source_counts['google_rss']}, newsapi={source_counts['newsapi']}) | "
            f'total_rows={len(all_rows):,} | ETA={eta_str}'
        )

    time.sleep(SLEEP_SEC)

news_df = pd.DataFrame(all_rows)
if len(news_df) == 0:
    raise RuntimeError('No news rows were collected. Check network/API settings and retry.')
raw_row_count = len(news_df)

news_df['Name'] = news_df['Name'].astype(str).str.strip()
news_df['date'] = pd.to_datetime(news_df['date'], errors='coerce').dt.normalize()
news_df['headline'] = news_df['headline'].astype(str).map(normalize_text)
if 'source' not in news_df.columns:
    news_df['source'] = 'unknown'
if 'url' not in news_df.columns:
    news_df['url'] = ''
if 'published_at' not in news_df.columns:
    news_df['published_at'] = ''
if 'query_symbol' not in news_df.columns:
    news_df['query_symbol'] = news_df['Name']
if 'query_text' not in news_df.columns:
    news_df['query_text'] = news_df['Name']

news_df = news_df.dropna(subset=['Name', 'date', 'headline'])
news_df = news_df[news_df['Name'].isin(symbols)].copy()
news_df['ingested_at'] = pd.Timestamp.now(tz='UTC').normalize().tz_localize(None)
news_df = news_df.sort_values(['Name', 'date', 'source', 'headline']).reset_index(drop=True)
news_df_full = news_df.copy()
news_df = news_df[(news_df['date'] >= window_start) & (news_df['date'] <= window_end)].copy()

if len(news_df) == 0:
    raise RuntimeError('Rows were collected, but all were removed by required-field cleaning.')

news_df_full.to_csv(raw_full_path, index=False)
news_df_export = news_df[['Name', 'date', 'headline', 'source']].copy()
news_df_export['date'] = pd.to_datetime(news_df_export['date'], errors='coerce').dt.strftime('%Y-%m-%d')
news_df_export.to_csv(raw_news_path, index=False)

log(f'[OK] Saved shared raw news to: {raw_news_path}')
log(f'[OK] Saved full unfiltered raw news to: {raw_full_path}')
log(f'Fetched raw rows before cleaning: {raw_row_count:,}')
log(f'Rows after required-field cleaning (full): {len(news_df_full):,}')
log(f'Rows (windowed): {len(news_df):,}')
log(f'Rows (exported 4-column): {len(news_df_export):,}')
log(f'Coverage: {news_df["date"].min().date()} -> {news_df["date"].max().date()}')
log('Source counts (top 10):')
print(news_df_export['source'].value_counts(dropna=False).head(10))
display(news_df_export.head(10))
log('=== Full-universe one-shot news ingestion finished ===')

[2026-04-02 18:48:23] === Full-universe one-shot news ingestion started ===
[2026-04-02 18:48:23] Universe size: 505 symbols
[2026-04-02 18:48:23] Price coverage: 2013-02-08 -> 2018-02-07
[2026-04-02 18:48:23] News coverage target: 2013-01-09 -> 2018-02-07
[2026-04-02 18:48:23] NEWS_BUFFER_DAYS=30
[2026-04-02 18:48:23] USE_NEWSAPI=True, API key provided=False, effective NewsAPI fetch=False
[2026-04-02 18:48:23] USE_GOOGLE_RSS=True, USE_YFINANCE=True
[2026-04-02 18:48:23] Output CSV: \\compdrive\Student5\25012923g\COMProfile\Documents\GitHub\ML-in-Finance-Data-Project\project_folder\news_data\data\news_headlines_raw.csv
[2026-04-02 18:48:41] Progress 20/505 | current=AGN | got=31 (yf=0, rss=31, newsapi=0) | total_rows=773 | ETA=2026-04-02 18:56:02
[2026-04-02 18:49:00] Progress 40/505 | current=ANDV | got=3 (yf=0, rss=3, newsapi=0) | total_rows=1,417 | ETA=2026-04-02 18:56:07
[2026-04-02 18:49:17] Progress 60/505 | current=BA | got=37 (yf=0, rss=37, newsapi=0) | total_rows=1,805 | ETA=2

,Name,date,headline,source
0,A,2013-02-18,George Washington could have held this stock - CNBC,google_rss
1,A,2013-02-19,5 Characteristics of the Perfect Momentum Stock - Medical Economics,google_rss
2,A,2013-03-05,Tocqueville's Francois Sicart: When Should a Value Investor Sell a Stock? - Barron's,google_rss
3,A,2013-06-04,Review Of Mark Minervini's 'Trade Like a Stock Market Wizard' - Seeking Alpha,google_rss
4,A,2013-06-25,Turning Spending into Savings: How One Emirati Teen Became a Stock Market Whiz - Knowledge at Wharton,google_rss
5,A,2013-07-22,How to Handle a Stock Stampede - Barron's,google_rss
6,A,2013-09-26,Calculating A Stock's Fair Value Based On Future Growth Expectations: Part 2A - Seeking Alpha,google_rss
7,A,2013-10-04,"Amid bloodshed in Pakistan, a stock exchange soars - CNBC",google_rss
8,A,2013-10-07,Why America Needs a Stock-Market Crash - The New Yorker,google_rss
9,A,2013-10-08,A stock-picking cat? Don’t LOL - MarketWatch,google_rss


[2026-04-02 18:56:16] === Full-universe one-shot news ingestion finished ===
